# Clasificación de Especies de Hongos - Optimizado para Multi-GPU

Optimizado para Kaggle con 2x T4 GPUs usando:
- Mixed Precision Training (AMP)
- DataParallel para multi-GPU
- Optimized DataLoaders con prefetching
- Gradient accumulation
- Pesos completos para portabilidad

## Preparación de Entorno

In [ ]:
path = '/kaggle/input/mushroom1'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
import timm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from tqdm.auto import tqdm
import warnings
import os
import time
import json
import gc

warnings.filterwarnings('ignore')

In [ ]:
# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False  # False para mejor performance
    torch.backends.cudnn.benchmark = True  # Optimiza convoluciones

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()

print(f"\n{'='*70}")
print(f"SYSTEM CONFIGURATION")
print(f"{'='*70}")
print(f"Device: {device}")
print(f"Number of GPUs: {NUM_GPUS}")

if torch.cuda.is_available():
    for i in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  - Memory: {props.total_memory / 1e9:.2f} GB")
        print(f"  - Compute Capability: {props.major}.{props.minor}")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")
print(f"{'='*70}\n")

## Configuración Global

In [ ]:
# ============== CONFIGURACIÓN ==============
# Imagen
IMG_SIZE = 224

# Batch size per GPU - T4 tiene 16GB, podemos usar batches grandes
# Con mixed precision podemos duplicar el batch size
BATCH_SIZE_PER_GPU = 64
BATCH_SIZE = BATCH_SIZE_PER_GPU * max(1, NUM_GPUS)  # 128 total para 2 GPUs

# Gradient accumulation para simular batches aún más grandes
GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch = 256

# Workers para DataLoader (2 workers por GPU es bueno en Kaggle)
NUM_WORKERS = 4 * max(1, NUM_GPUS)

# Épocas
NUM_EPOCHS_BASELINE = 20
NUM_EPOCHS = 25

# Learning rate (escalar con batch size efectivo)
BASE_LR = 3e-4
EFFECTIVE_BATCH = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LR = BASE_LR * (EFFECTIVE_BATCH / 64)  # Linear scaling rule
LR = min(LR, 1e-3)  # Cap max LR

# Mixed Precision
USE_AMP = True

# Dataset - USAR TODO EL DATASET
TRAIN_SUBSET_FRACTION = 1.0  # 100% del dataset
VAL_SUBSET_FRACTION = 1.0

print(f"Batch size per GPU: {BATCH_SIZE_PER_GPU}")
print(f"Total batch size: {BATCH_SIZE}")
print(f"Gradient accumulation steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {EFFECTIVE_BATCH}")
print(f"Learning rate: {LR:.6f}")
print(f"Num workers: {NUM_WORKERS}")
print(f"Mixed Precision: {USE_AMP}")

## Carga de Datasets

In [ ]:
train_csv = os.path.join(path, 'train.csv')
val_csv = os.path.join(path, 'val.csv')
test_csv = os.path.join(path, 'test.csv')

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

img_col = train_df.columns[0]
label_col = train_df.columns[1]

# Aplicar subset si es necesario (para debugging)
if TRAIN_SUBSET_FRACTION < 1.0:
    train_df = train_df.sample(frac=TRAIN_SUBSET_FRACTION, random_state=SEED).reset_index(drop=True)
if VAL_SUBSET_FRACTION < 1.0:
    val_df = val_df.sample(frac=VAL_SUBSET_FRACTION, random_state=SEED).reset_index(drop=True)

print(f"\n{'='*70}")
print(f"Estadísticas del Dataset")
print(f"{'='*70}")
print(f"Train: {len(train_df):,}")
print(f"Val: {len(val_df):,}")
print(f"Test: {len(test_df):,}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df):,}")
print(f"Columna de imagen: '{img_col}'")
print(f"Columna de etiqueta: '{label_col}'")
print(f"{'='*70}\n")

## Preparar Etiquetas

In [ ]:
# Combinar todas las etiquetas
all_labels = pd.concat([
    train_df[label_col],
    val_df[label_col],
    test_df[label_col]
])

# Encoder
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

num_classes = len(label_encoder.classes_)

# Guardar mapeo de clases para uso posterior en GUI
class_names = list(label_encoder.classes_)
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for idx, name in enumerate(class_names)}

print(f"\n{'='*70}")
print(f"Información de Especies")
print(f"{'='*70}")
print(f"Especies totales: {num_classes}")
print(f"\nPrimeras 10 especies:")
for i, species in enumerate(label_encoder.classes_[:10], 1):
    count = (train_df[label_col] == species).sum()
    print(f"{i:2d}. {species:45s} | {count:,} samples")
if num_classes > 10:
    print(f"... y {num_classes - 10} especies más")
print(f"{'='*70}\n")

## Dataset Class Optimizado

In [ ]:
class MushroomDataset(Dataset):
    """
    Dataset optimizado para multi-GPU training.
    - Pre-carga paths válidos
    - Cache de rutas
    - Manejo robusto de errores
    """
    def __init__(self, dataframe, root_dir, img_col, label_col,
                 label_encoder, transform=None, verify_images=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.img_col = img_col
        self.label_col = label_col
        self.label_encoder = label_encoder
        self.transform = transform
        
        # Pre-compute encoded labels
        self.labels = label_encoder.transform(self.dataframe[label_col])
        
        # Pre-compute image paths
        self.image_paths = []
        for idx in range(len(self.dataframe)):
            img_path_from_csv = self.dataframe.iloc[idx][img_col]
            species_name = self.dataframe.iloc[idx][label_col]
            filename = os.path.basename(img_path_from_csv)
            
            # Try different path strategies
            possible_paths = [
                os.path.join(root_dir, 'merged_dataset', species_name, filename),
                os.path.join(root_dir, species_name, filename),
            ]
            
            found_path = None
            for p in possible_paths:
                if os.path.exists(p):
                    found_path = p
                    break
            
            self.image_paths.append(found_path)
        
        # Placeholder image para errores
        self._placeholder = None
    
    def _get_placeholder(self):
        if self._placeholder is None:
            self._placeholder = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(128, 128, 128))
        return self._placeholder.copy()
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        try:
            path = self.image_paths[idx]
            
            if path is not None:
                image = Image.open(path).convert('RGB')
            else:
                image = self._get_placeholder()
            
            label = self.labels[idx]
            
            if self.transform:
                image = self.transform(image)
            
            return image, label
            
        except Exception as e:
            # Fallback silencioso
            placeholder = self._get_placeholder()
            if self.transform:
                placeholder = self.transform(placeholder)
            return placeholder, self.labels[idx] if idx < len(self.labels) else 0

## Transformaciones

In [ ]:
# Transformaciones para training con más augmentation
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),  # Resize más grande para random crop
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),  # Cutout-like augmentation
])

# Transformaciones para test y validation
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

## Datasets y DataLoaders Optimizados

In [ ]:
print("Creando datasets...")
train_dataset = MushroomDataset(
    train_df, path, img_col, label_col, label_encoder, train_transforms
)
val_dataset = MushroomDataset(
    val_df, path, img_col, label_col, label_encoder, val_transforms
)
test_dataset = MushroomDataset(
    test_df, path, img_col, label_col, label_encoder, val_transforms
)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Val dataset: {len(val_dataset):,} samples")
print(f"Test dataset: {len(test_dataset):,} samples")

In [ ]:
# DataLoaders optimizados para multi-GPU
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,  # Acelera transferencia CPU->GPU
    drop_last=True,   # Evita batches pequeños al final
    persistent_workers=True if NUM_WORKERS > 0 else False,  # Mantiene workers vivos
    prefetch_factor=2 if NUM_WORKERS > 0 else None,  # Pre-carga batches
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,  # Podemos usar batch más grande en eval
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
)

print(f"\nDataLoader configuración:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## Visualizar Imágenes de Ejemplo

In [ ]:
def show_batch(loader, label_encoder, n_images=8, title="Sample Images"):
    dataiter = iter(loader)
    images, labels = next(dataiter)
    
    # Denormalizar
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    images = images * std + mean
    images = torch.clamp(images, 0, 1)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    for idx, ax in enumerate(axes.flat):
        if idx < min(n_images, len(images)):
            ax.imshow(images[idx].permute(1, 2, 0))
            species = label_encoder.inverse_transform([labels[idx].item()])[0]
            ax.set_title(species, fontsize=9)
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visualizando muestras de entrenamiento...")
show_batch(train_loader, label_encoder, title="Training Samples (con augmentation)")

## Definir Modelos

In [ ]:
class BaselineCNN(nn.Module):
    """
    Baseline CNN simple para comparación.
    """
    def __init__(self, num_classes):
        super(BaselineCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Bloque 1: 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Bloque 2: 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Bloque 3: 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Bloque 4: 28 -> 14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Bloque 5: 14 -> 7
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
class MushroomClassifier(nn.Module):
    """
    Clasificador basado en EfficientNet con configuración optimizada.
    Guarda metadata para portabilidad.
    """
    def __init__(self, num_classes, model_name='efficientnet_b0', pretrained=True):
        super(MushroomClassifier, self).__init__()
        
        self.model_name = model_name
        self.num_classes = num_classes
        
        # Cargar modelo base
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes,
            drop_rate=0.3,
            drop_path_rate=0.2,
        )
    
    def forward(self, x):
        return self.backbone(x)
    
    def get_config(self):
        """Retorna configuración del modelo para guardado."""
        return {
            'model_name': self.model_name,
            'num_classes': self.num_classes,
        }

In [ ]:
def create_model(model_class, num_classes, device, **kwargs):
    """
    Crea modelo y lo envuelve en DataParallel si hay múltiples GPUs.
    """
    model = model_class(num_classes=num_classes, **kwargs)
    
    if NUM_GPUS > 1:
        print(f"Usando DataParallel con {NUM_GPUS} GPUs")
        model = nn.DataParallel(model)
    
    model = model.to(device)
    
    # Contar parámetros
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    return model

## Funciones de Entrenamiento con AMP

In [ ]:
def train_epoch_amp(model, loader, criterion, optimizer, scaler, device, 
                    epoch, num_epochs, accumulation_steps=1):
    """
    Entrenamiento de una época con Mixed Precision y Gradient Accumulation.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1:02d}/{num_epochs} [TRAIN]')
    optimizer.zero_grad()
    
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # Forward con mixed precision
        with autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss = loss / accumulation_steps  # Normalizar para accumulation
        
        # Backward con scaler
        scaler.scale(loss).backward()
        
        # Optimizer step cada accumulation_steps
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        # Estadísticas
        running_loss += loss.item() * accumulation_steps * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if batch_idx % 10 == 0:
            pbar.set_postfix({
                'loss': f'{running_loss/total:.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
@torch.no_grad()
def validate_amp(model, loader, criterion, device, phase='VAL'):
    """
    Validación con Mixed Precision.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f'{phase:5s}')
    
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': f'{running_loss/total:.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    val_loss = running_loss / total
    val_acc = 100. * correct / total
    
    return val_loss, val_acc

In [ ]:
def save_complete_checkpoint(model, optimizer, scheduler, scaler, epoch, 
                             val_acc, history, label_encoder, class_names,
                             filepath, is_best=False):
    """
    Guarda checkpoint completo con toda la información necesaria para:
    1. Continuar entrenamiento
    2. Usar en GUI/inferencia
    """
    # Obtener state dict del modelo (manejar DataParallel)
    if isinstance(model, nn.DataParallel):
        model_state = model.module.state_dict()
        model_config = model.module.get_config() if hasattr(model.module, 'get_config') else {}
    else:
        model_state = model.state_dict()
        model_config = model.get_config() if hasattr(model, 'get_config') else {}
    
    checkpoint = {
        # Estado del modelo
        'model_state_dict': model_state,
        'model_config': model_config,
        
        # Estado del entrenamiento
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'epoch': epoch,
        
        # Métricas
        'val_acc': val_acc,
        'history': history,
        
        # Información para inferencia/GUI
        'num_classes': len(class_names),
        'class_names': class_names,
        'class_to_idx': {name: idx for idx, name in enumerate(class_names)},
        'idx_to_class': {idx: name for idx, name in enumerate(class_names)},
        
        # Configuración de preprocesamiento (IMPORTANTE para GUI)
        'img_size': IMG_SIZE,
        'normalize_mean': [0.485, 0.456, 0.406],
        'normalize_std': [0.229, 0.224, 0.225],
        
        # Metadata
        'pytorch_version': torch.__version__,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    
    torch.save(checkpoint, filepath)
    
    if is_best:
        print(f"✓ Mejor modelo guardado: {filepath} (Val Acc: {val_acc:.2f}%)")

In [ ]:
def load_checkpoint_for_inference(filepath, device='cpu'):
    """
    Carga un checkpoint para inferencia (uso en GUI).
    Retorna el modelo listo para usar y la información necesaria.
    """
    checkpoint = torch.load(filepath, map_location=device)
    
    # Recrear el modelo
    model_config = checkpoint.get('model_config', {})
    num_classes = checkpoint['num_classes']
    
    if 'model_name' in model_config:
        # EfficientNet
        model = MushroomClassifier(
            num_classes=num_classes,
            model_name=model_config.get('model_name', 'efficientnet_b0'),
            pretrained=False
        )
    else:
        # Baseline CNN
        model = BaselineCNN(num_classes=num_classes)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    # Información para preprocesamiento
    preprocess_info = {
        'img_size': checkpoint.get('img_size', 224),
        'mean': checkpoint.get('normalize_mean', [0.485, 0.456, 0.406]),
        'std': checkpoint.get('normalize_std', [0.229, 0.224, 0.225]),
    }
    
    # Información de clases
    class_info = {
        'class_names': checkpoint.get('class_names', []),
        'idx_to_class': checkpoint.get('idx_to_class', {}),
        'num_classes': num_classes,
    }
    
    return model, preprocess_info, class_info

## Función de Entrenamiento Principal

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, 
                scheduler, num_epochs, device, model_name, class_names,
                label_encoder, accumulation_steps=1):
    """
    Loop de entrenamiento principal con todas las optimizaciones.
    """
    # Inicializar GradScaler para AMP
    scaler = GradScaler(enabled=USE_AMP)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'epoch_times': [], 'lr': []
    }
    
    best_val_acc = 0
    best_epoch = 0
    
    print(f"\n{'='*70}")
    print(f"INICIANDO ENTRENAMIENTO: {model_name}")
    print(f"{'='*70}")
    print(f"Total epochs: {num_epochs}")
    print(f"Training samples: {len(train_loader.dataset):,}")
    print(f"Validation samples: {len(val_loader.dataset):,}")
    print(f"Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * accumulation_steps})")
    print(f"Mixed Precision: {USE_AMP}")
    print(f"GPUs: {NUM_GPUS}")
    print(f"{'='*70}\n")
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]['lr']
        
        # Train
        train_loss, train_acc = train_epoch_amp(
            model, train_loader, criterion, optimizer, scaler,
            device, epoch, num_epochs, accumulation_steps
        )
        
        # Validate
        val_loss, val_acc = validate_amp(
            model, val_loader, criterion, device, phase='VAL'
        )
        
        # Update scheduler
        if scheduler:
            scheduler.step(val_acc)
        
        epoch_time = time.time() - epoch_start
        
        # Guardar historial
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_time)
        history['lr'].append(current_lr)
        
        # Print summary
        print(f"\n{'='*70}")
        print(f"EPOCH {epoch+1}/{num_epochs} SUMMARY")
        print(f"{'='*70}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
        print(f"Epoch Time: {epoch_time:.1f}s | LR: {current_lr:.6f}")
        
        # Guardar mejor modelo
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            
            save_complete_checkpoint(
                model, optimizer, scheduler, scaler,
                epoch, val_acc, history, label_encoder, class_names,
                f'{model_name}_best.pth', is_best=True
            )
        
        print(f"Best Val Acc: {best_val_acc:.2f}% (Epoch {best_epoch})")
        print(f"{'='*70}\n")
        
        # Guardar checkpoint periódico
        if (epoch + 1) % 5 == 0:
            save_complete_checkpoint(
                model, optimizer, scheduler, scaler,
                epoch, val_acc, history, label_encoder, class_names,
                f'{model_name}_epoch{epoch+1}.pth'
            )
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*70}")
    print(f"ENTRENAMIENTO COMPLETADO: {model_name}")
    print(f"{'='*70}")
    print(f"Tiempo total: {total_time/60:.1f} min")
    print(f"Tiempo promedio por época: {np.mean(history['epoch_times']):.1f}s")
    print(f"Mejor Val Accuracy: {best_val_acc:.2f}% (Epoch {best_epoch})")
    print(f"{'='*70}\n")
    
    return history, best_val_acc, best_epoch

## Entrenamiento del Modelo Baseline

In [ ]:
print("\n" + "="*70)
print("CREANDO MODELO BASELINE")
print("="*70)

baseline_model = create_model(BaselineCNN, num_classes, device)

baseline_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

baseline_optimizer = optim.AdamW(
    baseline_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

baseline_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    baseline_optimizer,
    mode='max',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [ ]:
baseline_history, best_baseline_acc, best_baseline_epoch = train_model(
    baseline_model, train_loader, val_loader,
    baseline_criterion, baseline_optimizer, baseline_scheduler,
    NUM_EPOCHS_BASELINE, device, 'baseline_cnn', class_names,
    label_encoder, accumulation_steps=GRADIENT_ACCUMULATION_STEPS
)

In [ ]:
# Evaluar en test
print("Evaluando baseline en test set...")
baseline_test_loss, baseline_test_acc = validate_amp(
    baseline_model, test_loader, baseline_criterion, device, phase='TEST'
)

print(f"\n{'='*70}")
print(f"BASELINE TEST RESULTS")
print(f"{'='*70}")
print(f"Test Accuracy: {baseline_test_acc:.2f}%")
print(f"Test Loss: {baseline_test_loss:.4f}")
print(f"{'='*70}\n")

In [ ]:
# Liberar memoria
baseline_params = sum(p.numel() for p in baseline_model.parameters())
baseline_total_time = sum(baseline_history['epoch_times'])

del baseline_model
del baseline_optimizer
del baseline_scheduler
del baseline_criterion

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print("Memoria liberada.")

## Entrenamiento de EfficientNet

In [ ]:
print("\n" + "="*70)
print("CREANDO MODELO EFFICIENTNET")
print("="*70)

model = create_model(
    MushroomClassifier, num_classes, device,
    model_name='efficientnet_b0', pretrained=True
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# Cosine Annealing es mejor para transfer learning
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=5,  # Restart cada 5 epochs
    T_mult=2,
    eta_min=1e-6
)

In [ ]:
# Usar ReduceLROnPlateau como alternativa
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [ ]:
history, best_val_acc, best_epoch = train_model(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    NUM_EPOCHS, device, 'efficientnet_mushroom', class_names,
    label_encoder, accumulation_steps=GRADIENT_ACCUMULATION_STEPS
)

In [ ]:
# Evaluar en test
print("Evaluando EfficientNet en test set...")
test_loss, test_acc = validate_amp(
    model, test_loader, criterion, device, phase='TEST'
)

print(f"\n{'='*70}")
print(f"EFFICIENTNET TEST RESULTS")
print(f"{'='*70}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print(f"{'='*70}\n")

In [ ]:
# Guardar modelo final completo
total_params = sum(p.numel() for p in model.parameters())
total_training_time = sum(history['epoch_times'])

# Guardar versión final con toda la información
save_complete_checkpoint(
    model, optimizer, scheduler, GradScaler(enabled=USE_AMP),
    NUM_EPOCHS - 1, test_acc, history, label_encoder, class_names,
    'efficientnet_mushroom_final.pth', is_best=True
)

print("Modelo final guardado con toda la información para GUI.")

## Visualización de Resultados

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Baseline CNN vs EfficientNet-B0 (Transfer Learning)',
             fontsize=16, fontweight='bold')

baseline_epochs = range(1, len(baseline_history['train_loss']) + 1)
efficientnet_epochs = range(1, len(history['train_loss']) + 1)

# Training Loss
axes[0, 0].plot(baseline_epochs, baseline_history['train_loss'],
                'b-o', label='Baseline', linewidth=2, markersize=4)
axes[0, 0].plot(efficientnet_epochs, history['train_loss'],
                'r-s', label='EfficientNet', linewidth=2, markersize=4)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Training Loss', fontsize=12)
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Validation Loss
axes[0, 1].plot(baseline_epochs, baseline_history['val_loss'],
                'b-o', label='Baseline', linewidth=2, markersize=4)
axes[0, 1].plot(efficientnet_epochs, history['val_loss'],
                'r-s', label='EfficientNet', linewidth=2, markersize=4)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Validation Loss', fontsize=12)
axes[0, 1].set_title('Validation Loss', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# Validation Accuracy
axes[1, 0].plot(baseline_epochs, baseline_history['val_acc'],
                'b-o', label='Baseline', linewidth=2, markersize=4)
axes[1, 0].plot(efficientnet_epochs, history['val_acc'],
                'r-s', label='EfficientNet', linewidth=2, markersize=4)
axes[1, 0].axhline(y=best_baseline_acc, color='b', linestyle='--', alpha=0.5,
                   label=f'Best Baseline: {best_baseline_acc:.1f}%')
axes[1, 0].axhline(y=best_val_acc, color='r', linestyle='--', alpha=0.5,
                   label=f'Best EfficientNet: {best_val_acc:.1f}%')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Validation Accuracy (%)', fontsize=12)
axes[1, 0].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Test Accuracy Bar Chart
models = ['Baseline\nCNN', 'EfficientNet\nB0']
test_accs = [baseline_test_acc, test_acc]
colors = ['#3498db', '#2ecc71']

bars = axes[1, 1].bar(models, test_accs, color=colors, alpha=0.8, width=0.6)
axes[1, 1].set_ylabel('Test Accuracy (%)', fontsize=12)
axes[1, 1].set_title('Final Test Accuracy', fontsize=14, fontweight='bold')
axes[1, 1].set_ylim(0, 100)
axes[1, 1].grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, test_accs):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 2,
                    f'{acc:.1f}%', ha='center', va='bottom',
                    fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('training_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Comparación Final

In [ ]:
print("="*70)
print("COMPARACIÓN FINAL")
print("="*70 + "\n")

comparison_data = {
    'Métrica': [
        'Arquitectura',
        'Parámetros',
        'Pesos Pre-entrenados',
        'Tiempo Total (min)',
        'Mejor Val Accuracy (%)',
        'Test Accuracy (%)',
        'Mejora vs Baseline'
    ],
    'Baseline CNN': [
        'CNN 5 capas',
        f'{baseline_params:,}',
        'No',
        f'{baseline_total_time/60:.1f}',
        f'{best_baseline_acc:.2f}',
        f'{baseline_test_acc:.2f}',
        'Baseline'
    ],
    'EfficientNet-B0': [
        'EfficientNet-B0',
        f'{total_params:,}',
        'Sí (ImageNet)',
        f'{total_training_time/60:.1f}',
        f'{best_val_acc:.2f}',
        f'{test_acc:.2f}',
        f'+{test_acc - baseline_test_acc:.2f}%'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("\n" + "="*70)

## Ejemplo de Uso para GUI

In [ ]:
# Este código muestra cómo cargar el modelo en una GUI
print("\n" + "="*70)
print("EJEMPLO DE CARGA PARA GUI")
print("="*70 + "\n")

example_code = '''
# En tu GUI, usa este código para cargar el modelo:

import torch
from torchvision import transforms
from PIL import Image

def load_model(checkpoint_path, device='cpu'):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Recrear modelo
    model = MushroomClassifier(
        num_classes=checkpoint['num_classes'],
        model_name=checkpoint['model_config']['model_name'],
        pretrained=False
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model, checkpoint

def predict(model, image_path, checkpoint, device='cpu'):
    # Preprocesamiento
    transform = transforms.Compose([
        transforms.Resize((checkpoint['img_size'], checkpoint['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=checkpoint['normalize_mean'],
            std=checkpoint['normalize_std']
        )
    ])
    
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        top5_probs, top5_indices = probs.topk(5)
    
    results = []
    idx_to_class = checkpoint['idx_to_class']
    for prob, idx in zip(top5_probs[0], top5_indices[0]):
        results.append({
            'species': idx_to_class[idx.item()],
            'probability': prob.item() * 100
        })
    
    return results

# Uso:
model, checkpoint = load_model('efficientnet_mushroom_best.pth')
predictions = predict(model, 'mi_hongo.jpg', checkpoint)
for pred in predictions:
    print(f"{pred['species']}: {pred['probability']:.1f}%")
'''

print(example_code)

In [ ]:
# Liberar memoria final
del model
del optimizer
del scheduler
del criterion

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Entrenamiento completado. Archivos guardados:")
print("  - baseline_cnn_best.pth")
print("  - efficientnet_mushroom_best.pth")
print("  - efficientnet_mushroom_final.pth")
print("  - training_comparison.png")